# A/B Test Analysis: Instant Book Feature Experiment

## Business context

Airbnb's booking flow has two modes — Instant Book lets guests 
book right away, Request to Book makes them wait for host 
approval. That wait is friction, and this experiment tests 
whether removing it increases occupancy.

Using Inside Airbnb's NYC data (May 2025). Instant Book isn't 
randomly assigned — hosts choose it — so I used propensity score 
matching in Notebook 2 to create fair control/treatment groups 
before testing anything here.

## Pre-registration

- H₀: Instant Book has no effect on occupancy rate
- H₁: Instant Book increases occupancy rate
- One-tailed t-test, α = 0.05, power = 0.80, MDE = 2.0pp
- Required n per group: 4,662 (matched dataset has 6,765)
- Primary metric: occupancy_rate
- CUPED covariate: number_of_reviews_ly — occupancy_rate itself 
  would be circular, and availability metrics are affected by 
  the treatment itself, so reviews-last-year was the best 
  non-circular, non-post-treatment option available (r=0.43 
  with occupancy)
- Subgroups: room type, borough, price tier — corrected with 
  Benjamini-Hochberg if multiple tests are run

**Limitation:** this is observational data — PSM controls for 
measured confounders only, not reverse causality or unobserved 
factors like host responsiveness. Results are associational, 
not strictly causal.

To avoid the peeking effect, nothing here gets adjusted once I 
start looking at results.

## Imports + load data

In [21]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')

path = "data/"
matched = pd.read_csv(path + "matched_listings.csv")

print(f'matched dataset: {matched.shape}')
print(matched['experiment_group'].value_counts())

matched dataset: (13530, 25)
experiment_group
treatment    6765
control      6765
Name: count, dtype: int64


In [23]:
# split group into control and treatment
control   = matched[matched['experiment_group'] == 'control'].copy()
treatment = matched[matched['experiment_group'] == 'treatment'].copy()

print(f'control (request to book) : {len(control):,}')
print(f'treatment (instant book)  : {len(treatment):,}')
print(f'\noccupancy rate:')
print(f'control mean   : {control["occupancy_rate"].mean():.4f}')
print(f'treatment mean : {treatment["occupancy_rate"].mean():.4f}')
print(f'raw difference : {treatment["occupancy_rate"].mean() - control["occupancy_rate"].mean():.4f}')

control (request to book) : 6,765
treatment (instant book)  : 6,765

occupancy rate:
control mean   : 0.1205
treatment mean : 0.1643
raw difference : 0.0438


## SRM check

Verify the matched groups are balanced 50/50 using a 
chi-squared test before any analysis.

In [24]:
# srm check — make sure the 50/50 split is real, not a fluke of how groups got assigned
def check_srm(n_control, n_treatment, expected_ratio=0.5):
    total = n_control + n_treatment
    expected_control   = total * expected_ratio
    expected_treatment = total * (1 - expected_ratio)

    chi2, p = stats.chisquare(
        [n_control, n_treatment],
        f_exp=[expected_control, expected_treatment]
    )

    verdict = 'PASS' if p >= 0.05 else 'FAIL'

    print(f'control      : {n_control:,}')
    print(f'treatment    : {n_treatment:,}')
    print(f'chi-squared  : {chi2:.4f}')
    print(f'p-value      : {p:.4f}')
    print(f'verdict      : {verdict}')

    return chi2, p

chi2, p_srm = check_srm(len(control), len(treatment))

control      : 6,765
treatment    : 6,765
chi-squared  : 0.0000
p-value      : 1.0000
verdict      : PASS


### SRM findings

Chi-squared = 0.0000, p-value = 1.0000 — PASS.

Groups are perfectly balanced at 6,765 each (expected from 
PSM 1:1 matching). No sample ratio mismatch — safe to proceed 
with CUPED and hypothesis testing.

## CUPED

Occupancy rate is highly variable (control std: 0.2266, 
treatment std: 0.2621) relative to the means (~0.12-0.16). 
CUPED reduces this variance using each listing's own 
occupancy rate as the pre-experiment covariate, increasing 
test sensitivity without needing more data.

In [25]:
# cuped — reduce variance using number_of_reviews_ly as the covariate
def apply_cuped(outcome, covariate):
    theta = np.cov(outcome, covariate)[0, 1] / np.var(covariate)
    adjusted = outcome - theta * (covariate - covariate.mean())
    var_reduction = 1 - np.var(adjusted) / np.var(outcome)
    print(f'theta              : {theta:.4f}')
    print(f'variance reduction : {var_reduction:.1%}')
    return adjusted

print('control:')
control['occupancy_adjusted'] = apply_cuped(
    control['occupancy_rate'],
    control['number_of_reviews_ly']
)

print('\ntreatment:')
treatment['occupancy_adjusted'] = apply_cuped(
    treatment['occupancy_rate'],
    treatment['number_of_reviews_ly']
)

raw_lift   = treatment['occupancy_rate'].mean() - control['occupancy_rate'].mean()
cuped_lift = treatment['occupancy_adjusted'].mean() - control['occupancy_adjusted'].mean()

print(f'\nraw lift           : {raw_lift:.4f}')
print(f'cuped-adjusted lift: {cuped_lift:.4f}')

control:
theta              : 0.0160
variance reduction : 46.7%

treatment:
theta              : 0.0027
variance reduction : 16.4%

raw lift           : 0.0438
cuped-adjusted lift: 0.0438


In [9]:
raw_lift   = treatment['occupancy_rate'].mean() - control['occupancy_rate'].mean()
cuped_lift = treatment['occupancy_adjusted'].mean() - control['occupancy_adjusted'].mean()

print(f'Raw lift          : {raw_lift:.4f}')
print(f'CUPED-adjusted lift: {cuped_lift:.4f}')

Raw lift          : 0.0438
CUPED-adjusted lift: 0.0438


### CUPED findings

Using number_of_reviews_ly as covariate (r=0.43 with 
occupancy_rate overall):

- Control: 46.7% variance reduction (theta=0.016)
- Treatment: 16.4% variance reduction (theta=0.003)

The asymmetry is interesting — control group occupancy is 
more strongly explained by historical review activity, 
while treatment group occupancy appears less tied to 
review history. This could suggest Instant Book listings' 
occupancy is driven more by the booking mechanism itself 
than by accumulated reputation.

Both groups show meaningful variance reduction, increasing 
the sensitivity of the hypothesis test below.

The CUPED-adjusted lift (0.0438) matches the raw lift exactly 
— this is expected, since CUPED preserves the mean while 
reducing variance. The benefit of CUPED appears in the 
hypothesis test below through a smaller standard error 
and larger t-statistic.

## Data Verification before hypothesis test

Quick verification that the matched dataset is correctly 
structured before running the main test.

In [26]:
# final sanity check before running the actual hypothesis test
print('1. group sizes')
print(f'   control   : {len(control):,} (required: 4,662)')
print(f'   treatment : {len(treatment):,} (required: 4,662)')

print('\n2. nulls, duplicates, overlap')
cols = ['occupancy_rate', 'number_of_reviews_ly', 'occupancy_adjusted']
print(f'   nulls control   : {control[cols].isnull().sum().sum()}')
print(f'   nulls treatment : {treatment[cols].isnull().sum().sum()}')
print(f'   duplicate ids   : {control["id"].duplicated().sum() + treatment["id"].duplicated().sum()}')
print(f'   group overlap   : {len(set(control["id"]) & set(treatment["id"]))}')

print('\n3. treatment/control assignment')
print(f'   control instant_bookable   : {control["instant_bookable"].unique()}')
print(f'   treatment instant_bookable : {treatment["instant_bookable"].unique()}')

print('\n4. occupancy rate range (should be 0-1)')
print(f'   control   : [{control["occupancy_rate"].min():.4f}, {control["occupancy_rate"].max():.4f}]')
print(f'   treatment : [{treatment["occupancy_rate"].min():.4f}, {treatment["occupancy_rate"].max():.4f}]')

print('\n5. extreme cuped-adjusted values (informational)')
ctrl_ext  = (control['occupancy_adjusted'] < -0.5).sum()
treat_ext = (treatment['occupancy_adjusted'] < -0.5).sum()
print(f'   control   : {ctrl_ext} listings ({ctrl_ext/len(control):.2%}) below -0.5')
print(f'   treatment : {treat_ext} listings ({treat_ext/len(treatment):.2%}) below -0.5')
print(f'   driven by extreme number_of_reviews_ly values, not excluded — see markdown note')

print('\nall checks complete — proceeding to hypothesis test')

1. group sizes
   control   : 6,765 (required: 4,662)
   treatment : 6,765 (required: 4,662)

2. nulls, duplicates, overlap
   nulls control   : 0
   nulls treatment : 0
   duplicate ids   : 0
   group overlap   : 0

3. treatment/control assignment
   control instant_bookable   : [False]
   treatment instant_bookable : [ True]

4. occupancy rate range (should be 0-1)
   control   : [0.0000, 0.6986]
   treatment : [0.0000, 0.6986]

5. extreme cuped-adjusted values (informational)
   control   : 7 listings (0.10%) below -0.5
   treatment : 7 listings (0.10%) below -0.5
   driven by extreme number_of_reviews_ly values, not excluded — see markdown note

all checks complete — proceeding to hypothesis test


# Hypothesis Test

In [27]:
def run_hypothesis_test(control_vals, treatment_vals, alpha=0.05):
    t_stat, p_value_two_tail = stats.ttest_ind(
        treatment_vals, control_vals, equal_var=False
    )
    p_value = p_value_two_tail / 2 if t_stat > 0 else 1 - p_value_two_tail / 2

    lift = treatment_vals.mean() - control_vals.mean()
    se = np.sqrt(treatment_vals.var()/len(treatment_vals) +
                 control_vals.var()/len(control_vals))
    ci_lower = lift - 1.96 * se
    ci_upper = lift + 1.96 * se

    print(f'control mean    : {control_vals.mean():.4f}')
    print(f'treatment mean  : {treatment_vals.mean():.4f}')
    print(f'lift            : {lift:.4f}')
    print(f't-statistic     : {t_stat:.4f}')
    print(f'p-value (1-tail): {p_value:.6f}')
    print(f'95% CI          : [{ci_lower:.4f}, {ci_upper:.4f}]')
    print(f'significant     : {"yes" if p_value < alpha else "no"}')

    return t_stat, p_value, lift


print('=== without cuped ===')
run_hypothesis_test(control['occupancy_rate'], treatment['occupancy_rate'])

print('\n=== with cuped ===')
run_hypothesis_test(control['occupancy_adjusted'], treatment['occupancy_adjusted'])

=== without cuped ===
control mean    : 0.1205
treatment mean  : 0.1643
lift            : 0.0438
t-statistic     : 10.3892
p-value (1-tail): 0.000000
95% CI          : [0.0355, 0.0520]
significant     : yes

=== with cuped ===
control mean    : 0.1205
treatment mean  : 0.1643
lift            : 0.0438
t-statistic     : 12.3625
p-value (1-tail): 0.000000
95% CI          : [0.0368, 0.0507]
significant     : yes


(np.float64(12.362524681933923),
 np.float64(3.395394115268118e-35),
 np.float64(0.04376838886695218))

In [28]:
# manual sanity check using scipy directly, no custom function
from scipy import stats

t_check, p_check = stats.ttest_ind(
    treatment['occupancy_rate'], 
    control['occupancy_rate'], 
    equal_var=False
)

print(f't-statistic (two-tailed): {t_check:.4f}')
print(f'p-value (two-tailed): {p_check:.2e}')
print(f'p-value (one-tailed): {p_check/2:.2e}')

t-statistic (two-tailed): 10.3892
p-value (two-tailed): 3.47e-25
p-value (one-tailed): 1.73e-25
